# Reading exercise: Section 4 answers (local, not for students)

Worked versions of the "Things to try" tasks in `reading.ipynb`. On a student clone, run `reading.ipynb` through Section 3 first. On the `answers` branch, the
second code cell builds Andrew's finished code and makes the input files itself. Use the phaindia kernel and open this
notebook from `reading/`.

Task 2 writes its own jsonnet copy (`hits_p120.jsonnet`) and output file, so it does not touch the
student's files.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

ROOT_LIBDIR = subprocess.check_output(["root-config", "--libdir"], text=True).strip()
if ROOT_LIBDIR not in sys.path:
    sys.path.insert(0, ROOT_LIBDIR)
import ROOT
from IPython.display import Image

READING_DIR = Path.cwd()
assert (READING_DIR / "hits_from_fixed_pedestal.jsonnet").exists(), "Open this notebook from the reading/ directory"
PREFIX = READING_DIR.parent / "opt"


def prepend(var, path):
    entries = [e for e in os.environ.get(var, "").split(":") if e]
    if str(path) not in entries:
        os.environ[var] = ":".join([str(path)] + entries)


for var, sub in [("LD_LIBRARY_PATH", "lib"), ("PATH", "bin"), ("PHLEX_PLUGIN_PATH", "lib")]:
    prepend(var, PREFIX / sub)
for lib in ("libwire_readout.so", "libhit.so"):
    ROOT.gSystem.Load(str(PREFIX / "lib" / lib))

HITS = "root_rntuple__window_hit_finder:window_hit_finder__hit_reco"
WIRES = "root_rntuple__wires_provider:simulated_wires__sim"
N_WIRES, N_APAS = 1024, 24


def hits_df(file_name):
    df = ROOT.RDataFrame(HITS, file_name)
    return (df.Alias("peak_adc", "window_hits.peak_adc_")
              .Define("width", "window_hits.end_time_tick_ - window_hits.start_time_tick_")
              .Define("n_hits", "peak_adc.size()"))


In [ ]:
# Build the tutorial and make both input files if they are missing.
# On the answers branch (Andrew's finished main + notebooks) this makes the notebook self-contained.
BUILD_DIR = PREFIX / "build"
if not (PREFIX / "lib" / "libhit.so").exists():
    BUILD_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cmake", str(READING_DIR.parent), f"-DCMAKE_INSTALL_PREFIX={PREFIX}", "-DCMAKE_BUILD_TYPE=Release"],
                   cwd=BUILD_DIR, check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["make", "-j4", "install"], cwd=BUILD_DIR, check=True, stdout=subprocess.DEVNULL)
    for lib in ("libwire_readout.so", "libhit.so"):
        ROOT.gSystem.Load(str(PREFIX / "lib" / lib))
for jsonnet, stage, out in [("../writing/write_wires.jsonnet", "sim", "form-tutorial-writing.root"),
                            ("hits_from_fixed_pedestal.jsonnet", "hit_reco", "form-tutorial-reading.root")]:
    if not Path(out).exists():
        subprocess.run(["phlex", "-c", jsonnet, "-j", "1", "--stage", stage],
                       check=True, stdout=open(f"{stage}.log", "w"), stderr=subprocess.STDOUT)
    print(out, "exists:", Path(out).exists())

df = hits_df("form-tutorial-reading.root")
print(f"{int(df.Sum('n_hits').GetValue())} hits")

## Task 1a: noise only, linear scale

In [ ]:
h = df.Histo1D(("noise", "Noise hits;Peak ADC;Hits", 45, 55, 100), "peak_adc")
c1 = ROOT.TCanvas("c1", "", 800, 500)
h.Draw()
c1.Print("task1a_noise.png")
Image("task1a_noise.png")

## Task 1b: signal only, and the signal fraction

In [ ]:
h = df.Histo1D(("signal", "Signal hits;Peak ADC;Hits", 100, 100, 3000), "peak_adc")
n_signal = df.Define("n_signal", "Sum(peak_adc > 100)").Sum("n_signal").GetValue()
n_all = df.Sum("n_hits").GetValue()
print(f"{int(n_signal)} of {int(n_all)} hits are above 100 ADC ({100 * n_signal / n_all:.2f}%)")

c1 = ROOT.TCanvas("c1b", "", 800, 500)
h.Draw()
c1.Print("task1b_signal.png")
Image("task1b_signal.png")

## Task 1c: hit width, and peak ADC against width

Noise hits are a few ticks wide and below 100 ADC. Signal hits are wider and taller, so they sit in a
separate band in the 2D plot.

In [ ]:
h_w = df.Histo1D(("width", "Hit width;Width (ticks);Hits", 100, 0, 100), "width")
h_2d = df.Histo2D(("peak_vs_width", "Peak ADC vs width;Width (ticks);Peak ADC", 100, 0, 100, 100, 50, 3000),
                  "width", "peak_adc")

c1 = ROOT.TCanvas("c1c", "", 1200, 500)
c1.Divide(2, 1)
c1.cd(1)
ROOT.gPad.SetLogy()
h_w.Draw()
c1.cd(2)
ROOT.gPad.SetLogz()
h_2d.Draw("colz")
c1.Print("task1c_width.png")
Image("task1c_width.png")

## Task 2: pedestal 60 against pedestal 120

Writes `hits_p120.jsonnet` from the current jsonnet (adds `pedestal: 120,` and renames the output),
runs it, and overlays the two peak-ADC distributions.

In [ ]:
text = Path("hits_from_fixed_pedestal.jsonnet").read_text()
text, n_ped = re.subn(r"(fixed_pedestal:\s*{\s*cpp:\s*'window_hit_finder_alg',)", r"\1\n      pedestal: 120,", text, count=1)
text, n_out = re.subn(r"form-tutorial-reading\.root", "form-tutorial-reading-p120.root", text)
assert n_ped == 1 and n_out == 1, "jsonnet does not look like the finished exercise"
Path("hits_p120.jsonnet").write_text(text)

!phlex -c hits_p120.jsonnet -j 1 --stage hit_reco > hits_p120.log 2>&1 || tail -20 hits_p120.log
!ls -l form-tutorial-reading-p120.root

In [ ]:
df120 = hits_df("form-tutorial-reading-p120.root")

binning = (100, 0, 3000)
h60 = df.Histo1D(("p60", "Peak ADC;Peak ADC;Hits", *binning), "peak_adc")
h120 = df120.Histo1D(("p120", "Peak ADC;Peak ADC;Hits", *binning), "peak_adc")

c2 = ROOT.TCanvas("c2", "", 800, 500)
c2.SetLogy()
h60.SetLineColor(ROOT.kBlue + 1)
h120.SetLineColor(ROOT.kOrange + 7)
h60.Draw("hist")
h120.Draw("hist same")
legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
legend.AddEntry(h60.GetPtr(), "pedestal 60", "l")
legend.AddEntry(h120.GetPtr(), "pedestal 120", "l")
legend.Draw()
c2.Print("task2_pedestals.png")

n_wires = N_WIRES * df.Count().GetValue()
for label, d in [("60", df), ("120", df120)]:
    n = d.Sum("n_hits").GetValue()
    print(f"pedestal {label:>3}: {int(n):>8} hits, {n / n_wires:6.2f} per wire")
Image("task2_pedestals.png")

At 120 the noise hits are gone, because the noise never goes above 100 ADC. What is left is the
signal pulses, and each one is a little narrower because less of it is above the higher pedestal.

## Task 3: hits on one waveform

Picks the tallest hit in APA 13, draws that wire's waveform from the writing tutorial's file (the same
way `plot_one_wire.py` does), and marks every hit on it at the pedestal height.

In [ ]:
APA, PEDESTAL = 13, 60

cols = ["window_hits.channel_", "window_hits.start_time_tick_", "window_hits.end_time_tick_", "window_hits.peak_adc_"]
one_apa = ROOT.RDataFrame(HITS, "form-tutorial-reading.root").Filter(f"rdfentry_ == {APA}")
data = one_apa.AsNumpy(cols)  # AsNumpy works out the column types itself
channel, start, end, peak = [list(data[c][0]) for c in cols]

best = max(range(len(peak)), key=lambda i: peak[i])
wire = channel[best]
on_wire = [(s, e) for ch, s, e in zip(channel, start, end) if ch == wire]
print(f"APA {APA}, channel {wire}: {len(on_wire)} hits, tallest peak {peak[best]} ADC")

reader = ROOT.RNTupleReader.Open(WIRES, "form-tutorial-writing.root")
view = reader.GetView["std::vector<uint16_t>"]("sim._0.waveform_")
waveform = list(view(wire + APA * N_WIRES))

hist = ROOT.TH1D("waveform", f"APA {APA}, channel {wire};Time tick;ADC", len(waveform), 0, len(waveform))
for tick, value in enumerate(waveform):
    hist.SetBinContent(tick + 1, value)

c3 = ROOT.TCanvas("c3", "", 1000, 500)
hist.Draw("hist")
lines = []
for s, e in on_wire:
    line = ROOT.TLine(s, PEDESTAL, e, PEDESTAL)
    line.SetLineColor(ROOT.kRed)
    line.SetLineWidth(4)
    line.Draw()
    lines.append(line)
c3.Print("task3_hits_on_wire.png")
del view, reader
Image("task3_hits_on_wire.png")